In [1]:
!/usr/local/cuda/bin/nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2021 NVIDIA Corporation
Built on Sun_Feb_14_21:12:58_PST_2021
Cuda compilation tools, release 11.2, V11.2.152
Build cuda_11.2.r11.2/compiler.29618528_0


In [8]:
!nvcc -o pso /content/cudaDE2022/main.cpp /content/cudaDE2022/DifferentialEvolution.cpp /content/cudaDE2022/DifferentialEvolutionGPU.cu

In [9]:
!make

make: *** No targets specified and no makefile found.  Stop.


In [10]:
!./pso

[
  {
    "dim": 10,
    "pop": 50,
    "gen": 1040,
    "CR": 0.900,
    "F": 0.800,
    "rnuns": 25,
    "x_mean":[ 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000],
    "x_std": [ 0.000,-0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000,-0.000,-0.000],
    "y_mean":  0.000,
    "y_std":   0.000
  },
  {
    "dim": 10,
    "pop": 100,
    "gen": 1040,
    "CR": 0.900,
    "F": 0.800,
    "rnuns": 25,
    "x_mean":[ 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000],
    "x_std": [ 0.000, 0.000, 0.000, 0.000,-0.000, 0.000, 0.000,-0.000, 0.000, 0.000],
    "y_mean":  0.000,
    "y_std":   0.000
  },
  {
    "dim": 10,
    "pop": 500,
    "gen": 1040,
    "CR": 0.900,
    "F": 0.800,
    "rnuns": 25,
    "x_mean":[ 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000],
    "x_std": [ 0.000, 0.000, 0.000,-0.000, 0.000,-0.000, 0.000, 0.000,-0.000, 0.000],
    "y_mean":  0.000,
    "y_std":   0.000
  },
  {
    "dim": 50,
    "pop": 50,

In [11]:
!nvcc -o programDE cudaDE2022/main.cpp cudaDE2022/DifferentialEvolution.cpp cudaDE2022/DifferentialEvolutionGPU.cu


In [12]:
!./programDE

[
  {
    "dim": 10,
    "pop": 50,
    "gen": 1040,
    "CR": 0.900,
    "F": 0.800,
    "rnuns": 25,
    "x_mean":[ 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000],
    "x_std": [ 0.000, 0.000, 0.000, 0.000,-0.000, 0.000,-0.000, 0.000, 0.000,-0.000],
    "y_mean":  0.000,
    "y_std":   0.000
  },
  {
    "dim": 10,
    "pop": 100,
    "gen": 1040,
    "CR": 0.900,
    "F": 0.800,
    "rnuns": 25,
    "x_mean":[ 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000],
    "x_std": [ 0.000,-0.000,-0.000,-0.000, 0.000, 0.000, 0.000, 0.000,-0.000, 0.000],
    "y_mean":  0.000,
    "y_std":   0.000
  },
  {
    "dim": 10,
    "pop": 500,
    "gen": 1040,
    "CR": 0.900,
    "F": 0.800,
    "rnuns": 25,
    "x_mean":[ 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000, 3.000],
    "x_std": [ 0.000, 0.000,-0.000, 0.000, 0.000,-0.000, 0.000, 0.000, 0.000, 0.000],
    "y_mean":  0.000,
    "y_std":   0.000
  },
  {
    "dim": 50,
    "pop": 50,

In [20]:
import subprocess, re, scipy, json, os, numpy as np, warnings; warnings.filterwarnings('ignore')
from pprint import pprint

# ----------------------------------------------------------------
def htest(testf, x,y, p=0.05):
	if np.allclose(np.array(x),np.array(y)):  print(f'\x1b[31mWARN  \x1b[0mthe {testf} test works not when x-y is the zero vector'); return  # (np.array(y)-np.array(x) != np.zeros(len(y))).sum()==0
	stat,p = testf(x,y)
	msg    = 'probably the same distribution' if 0.05<p else 'probably different distributions'
	print(f"\x1b[32m{str(testf).split()[1]:8}  \x1b[0m{msg:32}  stat {stat:6.3f}  p {p:.3f}")

def test(program='cudaDE2022/programDE'):
	print(f'\n----------------------------------------------------------------\n\x1b[35m{program}\x1b[0m')
	if not os.path.exists(program):  print(f"\x1b[91mFAIL  \x1b[92m{program} \x1b[0mnot found. Make sure that {program} has been compiled, and that it's running properly first");  return
	reply = subprocess.run(program, stdout=subprocess.PIPE, shell=True)
	try:     data  = json.loads(reply.stdout)
	except:
		print(reply.stdout.decode('utf-8'))
		print('\x1b[91mFAIL  \x1b[0mCUDA program execution failed. Make sure the programming is running okay')
		return

	for i in range(3):
		data0 = data[3*i+0]['x_mean']
		data1 = data[3*i+1]['x_mean']
		print()
		print(f"dim \x1b[31m{data[3*i]['dim']}  \x1b[0mpop \x1b[32m{data[3*i]['pop']}  \x1b[0mgen \x1b[34m{data[3*i]['gen']}\x1b[0m")
		print('data0', data0)
		print('data1', data1)
		htest(scipy.stats.wilcoxon, data0,data1)
		htest(scipy.stats.kruskal,  data0,data1)

# ----------------------------------------------------------------
test('cudaPSO2022/pso')
test('cudaDE2022/programDE')

# ----------------------------------------------------------------
# import numpy np
# data0 = np.random.normal(0,1, 0x10)  # np.random.normal(0,1, 0x10)
# data1 = np.random.normal(0,2, 0x10)  # np.random.normal(0,2, 0x10)
# htest(scipy.stats.wilcoxon, data0,data1)  # wilcoxon signed-rank test
# htest(scipy.stats.kruskal,  data0,data1)  # kruskal-wallis h     test



----------------------------------------------------------------
cudaPSO2022/pso

dim 10  pop 50  gen 1040
data0 [3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0]
data1 [3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0]
WARN  the <function wilcoxon at 0x7faf95a44790> test works not when x-y is the zero vector
WARN  the <function kruskal at 0x7faf95a185e0> test works not when x-y is the zero vector

dim 50  pop 50  gen 5200
data0 [3.001, 3.0, 3.0, 3.001, 3.0, 2.999, 3.0, 3.001, 3.0, 3.0, 3.0, 3.0, 2.999, 2.999, 3.0, 2.999, 3.0, 3.0, 3.0, 3.0, 3.001, 3.0, 3.0, 3.0, 3.0, 3.0, 2.999, 2.999, 3.001, 3.001, 3.001, 3.0, 2.999, 3.001, 3.0, 2.999, 3.0, 3.0, 2.999, 3.0, 3.0, 3.0, 3.0, 3.0, 3.001, 3.0, 3.001, 3.0, 3.001, 3.0]
data1 [2.947, 2.993, 3.011, 3.036, 2.995, 2.97, 2.953, 2.976, 2.995, 3.006, 2.993, 2.975, 2.95, 2.972, 2.945, 2.942, 2.997, 2.967, 2.951, 3.042, 2.961, 3.057, 3.053, 3.016, 3.056, 3.0, 3.006, 3.005, 3.017, 3.022, 3.019, 2.958, 2.978, 3.003, 2.987, 2.941, 3.024, 2.94, 3.033,